In [1]:
import pandas as pd
import numpy as np
import json
from sklearn.model_selection import train_test_split
files = ["data/orders_2024_01.csv", "data/orders_2024_02.csv", "data/orders_2024_03.csv"]

df_orders = pd.concat(
    [pd.read_csv(f, parse_dates=["order_datetime"]) for f in files],
    ignore_index=True
)
df_orders["revenue"] = df_orders["price"] * df_orders["quantity"]
# records-формат
with open("data/events_records.json", encoding="utf-8") as f:
    raw = json.load(f)

n = len(raw[0])
df_records = pd.DataFrame({
    "user_id":    [raw[1][str(i)] for i in range(n)],
    "event_type": [raw[3][str(i)] for i in range(n)],
})

# lines-формат
df_lines = pd.read_json("data/events_lines.json", lines=True)
df_lines = df_lines[["user_id", "event_type"]]

df_events = pd.concat([df_records, df_lines], ignore_index=True)
user_orders = df_orders.groupby("user_id").agg(
    orders_count =("order_id", "nunique"),
    revenue_sum  =("revenue",  "sum"),
    items_count  =("quantity", "sum"),
).reset_index()
user_events = df_events.groupby("user_id").agg(
    events_total       =("event_type", "count"),
    events_add_to_cart =("event_type", lambda s: (s == "add_to_cart").sum()),
    events_purchase    =("event_type", lambda s: (s == "purchase").sum()),
).reset_index()
user_features = user_orders.merge(user_events, on="user_id", how="outer").fillna(0)

# Средний чек
user_features["avg_check"] = np.where(
    user_features["orders_count"] > 0,
    user_features["revenue_sum"] / user_features["orders_count"],
    0
)

user_features.head()


   user_id  orders_count  revenue_sum  items_count  events_total  \
0     1001           177   4435767.09          454            54   
1     1002           123   2830004.52          281            41   
2     1003           156   4025266.18          389            39   
3     1004           147   3788030.56          380            37   
4     1005           141   3185653.37          346            44   

   events_add_to_cart  events_purchase     avg_check  
0                   7               16  25060.831017  
1                   7               12  23008.166829  
2                   6               16  25802.988333  
3                   8               15  25768.915374  
4                  10               13  22593.286312  


In [3]:
user_features["is_repeated_buyer"] = (
    user_features["orders_count"] > 1
).astype(int)

user_features["is_repeated_buyer"].value_counts()

is_repeated_buyer
1    100
Name: count, dtype: int64

In [4]:

df_model = user_features[user_features["orders_count"] >= 1].copy()

feature_cols = [
    "orders_count",
    "revenue_sum",
    "items_count",
    "avg_check",
    "events_total",
    "events_add_to_cart",
    "events_purchase",
]

X = df_model[feature_cols]
y = df_model["is_repeated_buyer"]

"X:", X.shape, " y:", y.shape

('X:', (100, 7), ' y:', (100,))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Собираем обратно в DataFrame
train_df = X_train.copy()
train_df["is_repeated_buyer"] = y_train.values

test_df = X_test.copy()
test_df["is_repeated_buyer"] = y_test.values

# Сохраняем
train_df.to_csv("train_repeated_buyer.csv", index=False)
test_df.to_csv("test_repeated_buyer.csv", index=False)

"Train:", train_df.shape
"Test :", test_df.shape